# CIS 536/736: Computer Graphics and Animation
## Lab 6a: Real-Time Data Visualization & Visual Encoding in Unity 6

--- 

### **Module 10 / Lecture 32 Hands-on Lab**
**Course:** CIS 536/736 Computer Graphics and Animation  
**Topic:** Implementing Mackinlay's Visual Encoding Hierarchy, Procedural Mesh Generation for 2D Plots, and GPU Instanced Data Marking in Unity 6  
**Associated Reading:** Kieran Healy, *Data Visualization*, Ch. 3 & 4; GAMES101 Lecture 20

--- 

## **Lab Overview & Objectives**

In this lab, you will translate the perceptual encoding principles from **Lecture 32** into high-performance, real-time graphics pipelines using Unity 6 (C# and Shader Graph / HLSL).

By the end of this lab, you will be able to:
1. **Classify Data Types & Aesthetics:** Map continuous quantitative data to high-accuracy visual channels (Position, Length) and nominal data to lower-accuracy channels (Color Hue) following **Mackinlay's Hierarchy**.
2. **Build Procedural UI/Graph Meshes:** Construct custom 2D line plots dynamically by populating `Mesh.vertices`, `Mesh.triangles`, and `Mesh.colors` in C# without relying on GameObject clutter.
3. **Execute Instanced Data Marking:** Use `Graphics.RenderMeshInstanced` to render $\ge 10,000$ quantitative data points in 3D world space at 60+ FPS.
4. **Implement Perceptually Uniform Shader Ramps:** Author a custom shader/material mapping normalized scalar quantities to perceptually uniform colormaps (e.g., Viridis) rather than naive sRGB interpolation.

--- 
## **Task 1: Pre-Lab Knowledge Check & Visual Encoding Setup**

Run the Python code block below to initialize your lab environment verification and verify data scale mappings.

In [ ]:
# Verification Script: Mackinlay Channel Accuracy & Normalization Utility
import numpy as np

def normalize_to_channel(data, target_min=0.0, target_max=1.0):
    """
    Scales continuous quantitative data array to normalized visual channel bounds [target_min, target_max].
    """
    d_min, d_max = np.min(data), np.max(data)
    if d_max == d_min:
        return np.full_like(data, target_min)
    return target_min + (data - d_min) * (target_max - target_min) / (d_max - d_min)

# Sample real-time system data: Frame Times (ms) across 10 seconds
np.random.seed(536)
frame_times = np.random.normal(loc=16.6, scale=3.2, size=100)
frame_times = np.clip(frame_times, 8.0, 45.0)

# Map to Position Channel Y (Common Scale: [0.0, 5.0] meters/units)
y_positions = normalize_to_channel(frame_times, 0.0, 5.0)

print(f"[Lab Check] Processed {len(frame_times)} frame time samples.")
print(f"Raw Frame Times (ms) -> Min: {np.min(frame_times):.2f}, Max: {np.max(frame_times):.2f}")
print(f"Mapped Y-Positions   -> Min: {np.min(y_positions):.2f}, Max: {np.max(y_positions):.2f}")

--- 
## **Task 2: Procedural 2D Line Plot Mesh Generation (Unity C#)**

Create a new C# script named `ProceduralLinePlot.cs` in your Unity 6 project (`Assets/Scripts/Vis/`). Attach this component to a GameObject containing a `MeshFilter` and `MeshRenderer`.

This script dynamically generates a continuous line strip mesh of uniform width $W$ mapping data array indices to **Position X** (common scale) and quantitative metric values to **Position Y** (common scale).

In [ ]:
/* 
 * File: Assets/Scripts/Vis/ProceduralLinePlot.cs
 * Course: CIS 536/736 - Lab 6a
 */

using UnityEngine;

[RequireComponent(typeof(MeshFilter), typeof(MeshRenderer))]
public class ProceduralLinePlot : MonoBehaviour
{
    [Header("Plot Bounds & Geometry Configuration")]
    public float plotWidth = 10.0f;
    public float plotHeight = 4.0f;
    public float lineWidth = 0.08f;
    public Color plotColor = Color.cyan;

    private Mesh _lineMesh;

    public void GeneratePlot(float[] quantitativeData)
    {
        if (quantitativeData == null || quantitativeData.Length < 2) return;

        int pointCount = quantitativeData.Length;
        int vertexCount = pointCount * 2;
        int triangleCount = (pointCount - 1) * 6;

        Vector3[] vertices = new Vector3[vertexCount];
        int[] triangles = new int[triangleCount];
        Color[] colors = new Color[vertexCount];
        Vector2[] uvs = new Vector2[vertexCount];

        // Determine Data Extents for Normalization
        float minVal = float.MaxValue, maxVal = float.MinValue;
        foreach (float val in quantitativeData)
        {
            if (val < minVal) minVal = val;
            if (val > maxVal) maxVal = val;
        }
        float range = Mathf.Approximately(maxVal, minVal) ? 1.0f : (maxVal - minVal);

        // Populate Vertex & Channel Buffers
        int vertIdx = 0;
        int triIdx = 0;

        for (int i = 0; i < pointCount; i++)
        {
            // Mackinlay Rank 1: Position along common X and Y scales
            float normX = (float)i / (pointCount - 1);
            float normY = (quantitativeData[i] - minVal) / range;

            float posX = normX * plotWidth;
            float posY = normY * plotHeight;

            // Generate quad ribbon vertices (Top & Bottom offset)
            vertices[vertIdx] = new Vector3(posX, posY + (lineWidth * 0.5f), 0.0f);
            vertices[vertIdx + 1] = new Vector3(posX, posY - (lineWidth * 0.5f), 0.0f);

            colors[vertIdx] = plotColor;
            colors[vertIdx + 1] = plotColor;

            uvs[vertIdx] = new Vector2(normX, 1.0f);
            uvs[vertIdx + 1] = new Vector2(normX, 0.0f);

            // Construct Quad Triangles
            if (i < pointCount - 1)
            {
                triangles[triIdx++] = vertIdx;
                triangles[triIdx++] = vertIdx + 2;
                triangles[triIdx++] = vertIdx + 1;

                triangles[triIdx++] = vertIdx + 1;
                triangles[triIdx++] = vertIdx + 2;
                triangles[triIdx++] = vertIdx + 3;
            }

            vertIdx += 2;
        }

        // Upload Mesh to GPU
        if (_lineMesh == null)
        {
            _lineMesh = new Mesh { name = "Procedural_Data_Plot" };
            GetComponent<MeshFilter>().sharedMesh = _lineMesh;
        }
        else
        {
            _lineMesh.Clear();
        }

        _lineMesh.vertices = vertices;
        _lineMesh.triangles = triangles;
        _lineMesh.colors = colors;
        _lineMesh.uv = uvs;
        _lineMesh.RecalculateBounds();
    }
}

--- 
## **Task 3: High-Density GPU Instanced Scatter Plot (Unity C#)**

To render thousands of quantitative points without CPU overhead, implement `GPUInstancedScatterPlot.cs`. This script uses `Graphics.RenderMeshInstanced` to submit batch transformation matrices directly to Unity 6's instanced rendering pipeline.

In [ ]:
/* 
 * File: Assets/Scripts/Vis/GPUInstancedScatterPlot.cs
 * Course: CIS 536/736 - Lab 6a
 */

using UnityEngine;

public class GPUInstancedScatterPlot : MonoBehaviour
{
    [Header("Instance Setup")]
    public Mesh instanceMesh;
    public Material instanceMaterial; // Must have 'Enable GPU Instancing' checked
    [Range(100, 50000)] public int totalDataPoints = 10000;

    private Matrix4x4[] _matrices;
    private MaterialPropertyBlock _propertyBlock;
    private float[] _scalarValues;

    void Start()
    {
        InitializeDataset();
    }

    void InitializeDataset()
    {
        _matrices = new Matrix4x4[totalDataPoints];
        _scalarValues = new float[totalDataPoints];
        Vector4[] colors = new Vector4[totalDataPoints];
        _propertyBlock = new MaterialPropertyBlock();

        for (int i = 0; i < totalDataPoints; i++)
        {
            // Generate synthetic 3D spatial distribution
            float x = Random.Range(-15.0f, 15.0f);
            float z = Random.Range(-15.0f, 15.0f);
            float normDistance = Mathf.Clamp01(Vector2.Distance(Vector2.zero, new Vector2(x, z)) / 15.0f);
            
            // Quantitative Channel: Y-Position maps to scalar density function
            float y = (1.0f - normDistance) * 6.0f + Random.Range(-0.5f, 0.5f);
            _scalarValues[i] = y;

            // Spatial Transformation Matrix
            Vector3 pos = transform.position + new Vector3(x, y, z);
            Quaternion rot = Quaternion.identity;
            Vector3 scale = Vector3.one * 0.12f;
            _matrices[i] = Matrix4x4.TRS(pos, rot, scale);

            // Color saturation mapping (Secondary Channel)
            float normY = Mathf.InverseTransformPoint(transform, pos).y / 6.0f;
            colors[i] = new Vector4(normY, 0.2f, 1.0f - normY, 1.0f);
        }

        // Bind custom per-instance properties
        _propertyBlock.SetVectorArray("_InstanceColor", colors);
    }

    void Update()
    {
        if (instanceMesh == null || instanceMaterial == null) return;

        // Submit batch rendering requests (Batches of 1023 per Draw Call)
        int batchSize = 1023;
        for (int i = 0; i < totalDataPoints; i += batchSize)
        {
            int currentBatchCount = Mathf.Min(batchSize, totalDataPoints - i);
            Matrix4x4[] batchMatrices = new Matrix4x4[currentBatchCount];
            System.Array.Copy(_matrices, i, batchMatrices, 0, currentBatchCount);

            Graphics.RenderMeshInstanced(
                new RenderParams(instanceMaterial) { propertyBlock = _propertyBlock },
                instanceMesh,
                0,
                batchMatrices,
                currentBatchCount
            );
        }
    }
}

--- 
## **Task 4: Perceptually Uniform Colormap HLSL Shader**

Create a custom Unlit Shader or Custom Function node in Unity 6 Shader Graph (`Assets/Shaders/PerceptualColormap.hlsl`) to prevent false visual boundaries caused by naive RGB interpolation.

In [ ]:
/* 
 * File: Assets/Shaders/PerceptualColormap.hlsl
 * Course: CIS 536/736 - Lab 6a
 * Polynomial approximation of Viridis colormap for perceptually uniform quantitative encoding
 */

#ifndef PERCEPTUAL_COLORMAP_INCLUDED
#define PERCEPTUAL_COLORMAP_INCLUDED

void EvaluateViridis_float(float t, out float3 color)
{
    t = clamp(t, 0.0, 1.0);
    
    // Viridis 5th-order polynomial coefficients
    float3 c0 = float3(0.2777273, 0.0054073, 0.3340998);
    float3 c1 = float3(0.1050930, 1.4046135, 1.3845904);
    float3 c2 = float3(-0.3308618, 0.2148475, 0.0950951);
    float3 c3 = float3(-4.6342304, -2.0075786, -10.150048);
    float3 c4 = float3(6.2282554, 2.2274489, 15.719163);
    float3 c5 = float3(-2.4491107, -0.6397394, -7.3546757);

    color = c0 + t * (c1 + t * (c2 + t * (c3 + t * (c4 + t * c5))));
}

#endif

--- 
## **Lab 6a Deliverables & Submission Checklist**

To receive full credit for Lab 6a, submit the following artifacts to Canvas:

- [ ] **C# Code Files:** `ProceduralLinePlot.cs` and `GPUInstancedScatterPlot.cs` committed under `Assets/Scripts/Vis/`.
- [ ] **Shader Artifact:** `PerceptualColormap.hlsl` or compiled Shader Graph asset under `Assets/Shaders/`.
- [ ] **Performance Screenshot:** Unity Stats Window or Profiler showing $\ge 10,000$ scatter points rendering at $\ge 60$ FPS with $\le 5$ Draw Calls.
- [ ] **Short Written Reflection (1-2 Paragraphs):** Explain why mapping quantitative data to 3D cube volume (Volume Channel) introduces greater perceptual error than mapping to 2D Y-position (Position Channel) according to Mackinlay's hierarchy.